In [1]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error
)
import numpy as np
import re
import ast

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Lactate Manual Annotation Check

In [2]:
path_result = "../outputs/lactate_extraction_results_2026-04-02_14-10-58.csv"
path_result_chunk = "../outputs/lactate_extraction_results_chunks_2026-04-02_14-10-58.csv"

In [3]:
df = pd.read_csv(path_result)

In [4]:
df_true = df[df["llm_present"] == True]
df_false = df[df["llm_present"] == False]
df_null = df[df["llm_present"].isna()]

n_true = min(20, len(df_true))
n_false = min(20, len(df_false))
n_null = min(10, len(df_null))

sample_true = df_true.sample(n=n_true, random_state=42)
sample_false = df_false.sample(n=n_false, random_state=42)
sample_null = df_null.sample(n=n_null, random_state=42)

val_df = pd.concat([
    sample_true,
    sample_false,
    sample_null,
]).drop_duplicates(subset=["hadm_id"]).reset_index(drop=True)

In [5]:
val_df = val_df.drop(['structured_lactate_max', 'structured_label'], axis=1)

In [6]:
val_df.to_csv("lactate_validation_set.csv", index=False)

In [7]:
discharge_filtered = pd.read_csv("../../Outputs/discharge_filtered.csv")

### display both discharge note and its chunks

In [8]:
note_id = "16906282-DS-22"
#print(discharge_filtered.loc[discharge_filtered.note_id == note_id, 'text'].values[0])
#print(val_df.loc[val_df.note_id == note_id, 'chunks'].values[0])

## Shock Manual Annotation Check

In [40]:
path_result = "../outputs/shock_extraction_results_2026-04-10_21-57-35.csv" #"../outputs/shock_extraction_results_2026-04-02_16-36-31.csv"
path_result_chunk = "../outputs/shock_extraction_results_chunks_2026-04-10_21-57-35.csv" #"../outputs/shock_extraction_results_chunks_2026-04-02_16-36-31.csv" 
df = pd.read_csv(path_result)

In [44]:
df.loc[df.note_id == '17545621-DS-18', :]

,note_id,hadm_id,subject_id,note_type,llm_present,llm_severity,llm_evidence_quotes,llm_justification,llm_confidence,n_chunks,chunks,chunk_debug_rows
1210,17545621-DS-18,27382907,17545621,DS,NaN,NaN,[],No evidence of shock or hemodynamic instabilit...,1.0,1,['1. Heterogeneous enhancement of the liver ma...,"[{""chunk_index"": 0, ""chunk_text"": ""1. Heteroge..."


In [34]:
df.llm_severity.value_counts()

llm_severity
severe      815
mild        239
none        222
moderate    159
Name: count, dtype: int64

In [35]:
df.llm_present.value_counts()

llm_present
True     1226
False     222
Name: count, dtype: int64

In [36]:
df.shape

(1618, 12)

In [37]:
df.llm_present.isna().sum()

170

In [48]:
df.loc[df.n_chunks==0, :].shape

(162, 12)

In [32]:
df.loc[(df.llm_present.isna()) & (df.n_chunks != 0), :].to_csv('additional_nulls_shock.csv')

In [10]:
#df.loc[df.note_id=='12823860-DS-10', :]

In [11]:
df_shock_eval_old = pd.read_csv('shock_annotation_comp1 - shock_manual_annotation_sample.csv')
df_shock_eval_old_ids = df_shock_eval_old.note_id.to_list()

In [12]:
#df

In [13]:
df_first_part_old = df.loc[df.note_id.isin(df_shock_eval_old_ids), :].sort_values(by=['llm_present', 'llm_severity'], ascending=False)

In [14]:
df_first_part_old.head()
df_first_part_old["manual_shock_present"] = pd.NA
df_first_part_old["manual_shock_severity"] = pd.NA

In [15]:
df_first_part_old.llm_severity.value_counts()

llm_severity
severe      24
moderate    10
mild         9
none         9
Name: count, dtype: int64

In [16]:
df_first_part_old.shape

(52, 14)

In [17]:
df_second_part_new = df.loc[~df.note_id.isin(df_first_part_old.note_id), :]

In [18]:
df_second_part_new.llm_severity.value_counts()

llm_severity
severe      791
mild        230
none        213
moderate    149
Name: count, dtype: int64

In [19]:
# --- load ---
df = df_second_part_new #pd.read_csv(path_result)

# --- clean llm_present ---
def parse_bool(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, bool):
        return x
    x = str(x).strip().lower()
    if x == "true":
        return True
    if x == "false":
        return False
    return np.nan

df["llm_present"] = df["llm_present"].apply(parse_bool)

# --- split ---
df_true = df[df["llm_present"] == True].copy()
df_false = df[df["llm_present"] == False].copy()
df_null = df[df["llm_present"].isna()].copy()

# =====================================================
# 1. TRUE cases (balanced by severity)
# =====================================================
severity_target = {
    "severe": 0,
    "moderate": 5,
    "mild": 6
}

true_parts = []

for sev, n in severity_target.items():
    subset = df_true[df_true["llm_severity"] == sev]
    n_take = min(n, len(subset))  # safety
    if n_take > 0:
        true_parts.append(subset.sample(n=n_take, random_state=42))

sample_true = pd.concat(true_parts, ignore_index=True)

# =====================================================
# 2. FALSE cases
# =====================================================
n_false = min(6, len(df_false))
sample_false = df_false.sample(n=n_false, random_state=42) if n_false > 0 else pd.DataFrame()

# =====================================================
# combine
# =====================================================
val_df = pd.concat(
    [sample_true, sample_false],
    ignore_index=True
)

# ensure unique admissions
val_df = val_df.drop_duplicates(subset=["hadm_id"]).reset_index(drop=True)

# =====================================================
# keep columns
# =====================================================
cols_keep = [
    "note_id",
    "hadm_id",
    "subject_id",
    "note_type",
    "llm_present",
    "llm_severity",
    "llm_evidence_quotes",
    "llm_justification",
    "llm_confidence",
    "n_chunks",
    "chunks"
]

cols_keep = [c for c in cols_keep if c in val_df.columns]
val_df = val_df[cols_keep].copy()

# =====================================================
# manual annotation columns
# =====================================================
val_df["manual_shock_present"] = pd.NA
val_df["manual_shock_severity"] = pd.NA

# =====================================================
# combine
# =====================================================
val_df = pd.concat([df_first_part_old, val_df], axis=0, ignore_index=True)

# =====================================================
# sort (nice for annotation)
# =====================================================
val_df = val_df.sort_values(
    by=["llm_present", "llm_severity"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# =====================================================
# save
# =====================================================
out_path = "shock_manual_annotation_sample_new_prompt.csv"
val_df.to_csv(out_path, index=False)

# =====================================================
# sanity check
# =====================================================
print("Saved to:", out_path)
print("\nllm_present distribution:")
print(val_df["llm_present"].value_counts(dropna=False))

print("\nllm_severity (only True):")
print(val_df[val_df["llm_present"] == True]["llm_severity"].value_counts())

print("\nTotal rows:", len(val_df))

Saved to: shock_manual_annotation_sample_new_prompt.csv

llm_present distribution:
llm_present
True     54
False    15
Name: count, dtype: int64

llm_severity (only True):
llm_severity
severe      24
mild        15
moderate    15
Name: count, dtype: int64

Total rows: 69


In [38]:
note_id = "19124346-DS-10"
#print(discharge_filtered.loc[discharge_filtered.note_id == note_id, 'text'].values[0])
#print(val_df.loc[val_df.note_id == note_id, 'chunks'].values[0])

## Coma Manual Annotation Check

In [16]:
path_result = "../outputs/coma_extraction_results_2026-04-04_12-38-53.csv"
df = pd.read_csv(path_result)

In [17]:
# --- clean llm_present ---
def parse_bool(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, bool):
        return x
    x = str(x).strip().lower()
    if x == "true":
        return True
    if x == "false":
        return False
    return np.nan

df["llm_present"] = df["llm_present"].apply(parse_bool)

# --- split ---
df_true = df[df["llm_present"] == True]
df_false = df[df["llm_present"] == False]
df_null = df[df["llm_present"].isna()]

n_true = min(20, len(df_true))
n_false = min(20, len(df_false))
n_null = min(10, len(df_null))

sample_true = df_true.sample(n=n_true, random_state=42)
sample_false = df_false.sample(n=n_false, random_state=42)
sample_null = df_null.sample(n=n_null, random_state=42)

val_df = pd.concat([
    sample_true,
    sample_false,
    sample_null,
]).drop_duplicates(subset=["hadm_id"]).reset_index(drop=True)

In [18]:
val_df["manual_coma_present"] = pd.NA

out_path = "coma_manual_annotation_sample.csv"
#val_df

In [19]:
note_id = "10199560-DS-17"
#print(discharge_filtered.loc[discharge_filtered.note_id == note_id, 'text'].values[0])
#print(df_true.loc[df_true.note_id == note_id, 'chunks'].values[0])

### additional true cases

In [74]:
df_true.loc[~df_true.note_id.isin(sample_true.note_id), :].head(10).to_csv("coma_manual_annotation_sample_additionalTrue.csv", index=False)

### second iteration results for coma

In [25]:
second_coma = pd.read_csv('../outputs/coma_extraction_results_2026-04-04_18-31-04.csv')

In [26]:
#second_coma.loc[second_coma.note_id.isin(annotated_coma.note_id), :]

In [27]:
annotated_coma = pd.read_csv('coma_manual_annotation_sample - coma_manual_annotation_sample.csv')

In [28]:
#annotated_coma.sort_values(by='note_id')

In [29]:
df_merged = annotated_coma.merge(second_coma.loc[second_coma.note_id.isin(annotated_coma.note_id), :], how='left', suffixes=('_anno','_sec'), on='note_id')

In [30]:
#df_merged

In [31]:
df_merged.loc[:, ['note_id', 'llm_present_anno', 'llm_present_sec']] #second and first iteration results are the same for the already annotated set

,note_id,llm_present_anno,llm_present_sec
0,14357885-DS-15,True,True
1,11281603-DS-9,True,True
2,14265588-DS-10,True,True
3,16331210-DS-14,True,True
4,18752566-DS-10,True,True
5,19894518-DS-16,True,True
6,16022077-DS-31,True,True
7,15682089-DS-8,True,True
8,14368609-DS-19,True,True
9,11320106-DS-3,True,True


#### additional false cases from the second iteration

In [35]:
second_coma.loc[~second_coma.note_id.isin(annotated_coma.note_id) & (second_coma.llm_present == False), :].head(10).to_csv("coma_manual_annotation_sample_additionalFalse_secondIterration.csv", index=False)

In [36]:
note_id = "18341594-DS-19"
#print(discharge_filtered.loc[discharge_filtered.note_id == note_id, 'text'].values[0])
#print(second_coma.loc[second_coma.note_id == note_id, 'chunks'].values[0])